# 02 · Data cleaning and lineage
**Question:** What transformations turn raw AMLSim CSVs into analytical tables, and do they preserve every record?

Lineage: `SOURCE (github.com/IBM/AMLSim) → RAW (checksummed .tgz) → EXTRACTED CSV → CLEANED tables →
POINT-IN-TIME FEATURES → SCORES / ALERTS → FIGURES / DASHBOARD`. Nothing is dropped during cleaning: duplicates and
self-transfers are flagged, not deleted, because no transaction id exists to prove they are errors.

In [1]:
import sys, json, sqlite3
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 4)
T = lambda name: pd.read_csv(config.TABLES_DIR / f"{name}.csv")
KM = json.loads((config.TABLES_DIR / "key_metrics.json").read_text())
con = sqlite3.connect(config.DB_PATH)
def show(fig_name, width=11):
    img = plt.imread(config.FIGURES_DIR / f"{fig_name}.png")
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w)); ax.imshow(img); ax.axis("off")
print("outputs loaded from", config.TABLES_DIR.relative_to(ROOT))

outputs loaded from outputs/tables


In [2]:
from src import ingestion, preprocessing
nodes, tx_raw, _ = ingestion.load_raw("combined")
accounts, tx = preprocessing.clean(nodes, tx_raw)
assert len(tx) == len(tx_raw) and len(accounts) == len(nodes)
print("rows preserved:", len(tx), "transactions,", len(accounts), "accounts")
tx.head()

rows preserved: 120558 transactions, 20000 accounts


,tx_id,src,dst,amount,step,week,is_self_transfer,dup_rank,is_repeat_copy
0,1,216,14730,163.30,1,1,0,1,0
1,2,322,5431,143.11,1,1,0,1,0
2,3,78,19972,192.33,1,1,0,1,0
3,4,248,14820,101.86,1,1,0,1,0
4,5,242,18672,113.33,1,1,0,1,0


In [3]:
pd.Series({"exact duplicate extra copies (flagged)": int(tx.is_repeat_copy.sum()),
           "self-transfers (flagged)": int(tx.is_self_transfer.sum()),
           "accounts with no transactions": int(accounts.first_step.isna().sum()),
           "weeks covered": int(tx.week.max()),
           "value preserved": bool(np.isclose(tx.amount.sum(), tx_raw.value.sum()))})

,0
exact duplicate extra copies (flagged),1684
self-transfers (flagged),15
accounts with no transactions,20
weeks covered,22
value preserved,True


In [4]:
# the SQLite database built by the pipeline holds the same rows
pd.read_sql_query("SELECT COUNT(*) AS tx, SUM(is_repeat_copy) AS dup, SUM(is_self_transfer) AS self FROM transactions", con)

,tx,dup,self
0,120558,1684,15


In [5]:
# cross-checks between SQL and Python recorded by the pipeline
T("qc_sql_vs_python")

,check,sql_value,python_value,match
0,transactions,1.2056e+05,1.2056e+05,True
1,total value,3.3288e+07,3.3288e+07,True
2,accounts,2.0000e+04,2.0000e+04,True
3,labelled accounts,1.8040e+03,1.8040e+03,True
4,exact duplicate copies,1.6840e+03,1.6840e+03,True
5,self transfers,1.5000e+01,1.5000e+01,True
6,integrated alerts (dev),7.8400e+02,7.8400e+02,True
7,VEL-01 alerts (dev),1.1850e+03,1.1850e+03,True
8,integrated alerts (test),5.4000e+02,5.4000e+02,True
9,VEL-01 alerts (test),8.3700e+02,8.3700e+02,True


### Point-in-time feature table
For a monitoring run at the end of week *w* (step 7w) features use only transactions with step ≤ 7w:
this week's behaviour, the account's history before this week, and a 28-day network window.
`fraudStep` is carried as `fraud_step_raw` for transparency only; it is never a feature.

In [6]:
f = pd.read_sql_query("SELECT * FROM account_week_features LIMIT 5", con)
f.T.head(40)

,0,1,2,3,4
account_id,20,34,54,60,61
week,5,5,5,5,5
run_step,35,35,35,35,35
n_in,0.0,1.0,1.0,0.0,0.0
n_out,1.0,0.0,0.0,1.0,1.0
amt_in,0.0,461.22,249.99,0.0,0.0
amt_out,131.0,0.0,0.0,181.44,154.02
max_amt,131.0,461.22,249.99,181.44,154.02
mean_amt,131.0,461.22,249.99,181.44,154.02
n_cp,1,1,1,1,1


In [7]:
pd.read_sql_query("SELECT period, MIN(week) AS first_week, MAX(week) AS last_week, COUNT(*) AS account_weeks FROM account_week_features GROUP BY period", con)

,period,first_week,last_week,account_weeks
0,dev,5,12,78365
1,test,13,21,62085
